# Semaine 10 — Jour 3 : Résumé extractif, couverture et fidélité

Sélectionner des phrases originales sous un budget de mots.

Parcours essentiel : 60 minutes (cours 20 min, lab 25 min, exercice 10 min, synthèse 5 min). Prévoir 2 à 3 heures pour l’unité complète avec entretien et challenge. L’installation de l’environnement ajoute un temps distinct; le lab LoRA est optionnel.

Prérequis : Python 3.11+, bases de classification et de vecteurs. Le parcours utilise NumPy et scikit-learn, sans modèle à télécharger ni clé API. Installer les dépendances S10 selon [le guide](../../../docs/week10/QUICKSTART.md).

Lire [les objectifs](learning_objectives.md), [le cours](chapter.md), puis [le lab](labs/README.md). Tenter [les exercices](exercises.md), [l’entretien](interview.md) et [le challenge](challenge.md) avant les corrigés locaux.

Notebooks depuis la racine : `notebooks/week10/S10_J3_resume.ipynb` et `notebooks/week10/teacher/S10_J3_resume_teacher.ipynb`.

Critère de réussite : Distinguer similarité lexicale, couverture d’information et fidélité.

Les apprentissages du parcours principal sont réellement exécutables. Le corpus reste fictif, les baselines à règles sont identifiées et aucun résultat LoRA n’est annoncé.


# Objectifs mesurables

- Sélectionner des phrases originales sous un budget de mots.
- Conserver leurs positions et leur ordre source.
- Distinguer similarité lexicale, couverture d’information et fidélité.

## Preuves attendues

Une sortie reproductible, un invariant vérifié et une limite de la tâche correctement interprétée.


# S10 J3 — Résumé extractif, couverture et fidélité


## 1. Quel résumé produire ?
Un résumé peut aider à consulter un ticket, préparer une transmission ou réduire un contexte. Ces usages ne demandent pas les mêmes informations. Identifier les faits nécessaires : composant, symptôme, impact, action et incertitude. Un résumé court mais privé de la condition d’une action peut être dangereux même s’il reste fluide.

Le lab choisit un résumé extractif. Il reproduit des phrases du texte sans en générer de nouvelles. Cette méthode rend les références simples à vérifier, mais elle ne garantit pas que les phrases retenues sont les plus importantes ni que leur combinaison garde toutes les nuances.

## 2. Segmenter sans perdre les positions
sentence_spans sépare après une ponctuation suivie d’espace ou sur des lignes. Les décimales comme 1,5 et les adresses contenant un point sans espace restent dans leur segment. Les offsets pointent dans le texte original; chaque phrase retenue doit correspondre à source[start:end].

Ce segmentateur n’est pas un analyseur linguistique général. Des abréviations, dialogues ou listes peuvent le tromper. Les tests couvrent quelques formats du corpus, pas toutes les conventions du français. Modifier les règles demande une réévaluation des références et budgets.

## 3. Centralité TF-IDF
Chaque phrase est vectorisée. Le centroïde des vecteurs de phrases donne une représentation moyenne du document. La similarité avec ce centroïde classe les phrases par centralité lexicale. Ce calcul local non supervisé ne réutilise pas le vocabulaire du classifieur de J1 et n’utilise pas les résumés de référence pour choisir les phrases.

Une phrase répétant les termes fréquents peut être centrale sans contenir l’impact critique. Une phrase unique portant une négation importante peut être mal classée. La centralité est une heuristique à inspecter, pas une mesure de vérité ni d’importance universelle.

## 4. Budget et ordre
Le sélecteur conserve des phrases entières qui tiennent dans word_budget et s’arrête à max_sentences. Les phrases retenues sont ensuite remises dans l’ordre source pour faciliter la lecture. Un grand segment peut être sauté au profit d’un suivant plus court. Si rien ne tient, le statut budget_exhausted le signale.

Le budget est un nombre de mots séparés par espaces, pas un budget de tokens LLM. Ne pas comparer directement ce nombre aux tokens d’une API. Pour alimenter un modèle réel, compter aussi instructions, question, wrappers et réserve de sortie avec son tokenizer.

## 5. Mesurer ce que l’on sait mesurer
Le rapport fournit un chevauchement multiensemble de tokens avec une référence écrite. Ce proxy donne précision, rappel et F1 lexicaux. Ce n’est pas ROUGE, ce n’est pas une mesure de factualité et ce n’est pas une appréciation humaine d’utilité.

Le contrôle d’offsets prouve que chaque phrase existe dans la source. Il ne prouve pas que le résumé couvre toutes les affirmations importantes ni qu’il présente correctement leur relation. Une campagne plus utile annoterait les unités d’information et les omissions critiques, puis comparerait les méthodes sous le même protocole.

## 6. Du résumé extractif au génératif
Un modèle génératif peut reformuler et fusionner, mais ajoute un risque d’affirmations non soutenues. Garder le texte source, les références et une politique de revue. Le lab n’en exécute aucun; il prépare les contrôles que ce raccord devra conserver. Un extracteur de phrases n’est pas une preuve de « zéro hallucination » au sens d’une compréhension correcte.

## À retenir
La compression doit préserver le contrat d’information. La fidélité des slices, le recouvrement lexical et l’utilité du résumé sont trois questions différentes.


## Diagramme du jour

```mermaid
flowchart TD
 A["Texte et phrases exactes"] --> B["TF-IDF local"]
 B --> C["Centralité de phrase"]
 C --> D["Budget et sélection"]
 D --> E["Ordre source et résumé"]
 E --> F["Couverture à évaluer séparément"]
```

Le rendu Mermaid dépend du lecteur; la source .mmd reste disponible.

# Lab exécutable

NumPy et scikit-learn sont requis; aucune requête réseau dans les cellules.

In [ ]:
from pathlib import Path
import sys
cwd = Path.cwd().resolve()
root = next((p for p in (cwd, *cwd.parents) if (p / "examples/week10/nlp.py").is_file()), None)
assert root is not None, "Ouvrir le notebook depuis le dépôt décompressé"
sys.path.insert(0, str(root))
from examples.week10.nlp import summarize, read_data
example = read_data("examples.json")[0]
result = summarize(example["text"], max_sentences=2, word_budget=45)
assert result["word_count"] <= 45
assert len(result["sentences"]) <= 2
assert all(example["text"][s["start"]:s["end"]] == s["text"] for s in result["sentences"])
assert [s["start"] for s in result["sentences"]] == sorted(s["start"] for s in result["sentences"])
print(result["summary"])

# Lab J3 — Résumé extractif, couverture et fidélité

Depuis la racine du dépôt, dans l’environnement S10 :

```bash
python3 book/week10/day03/labs/run.py
python3 -m examples.week10.demo --day 3
```

Résultat attendu : **phrases source, budget respecté et chevauchement lexical de référence**.

## Travail

1. Exécuter le scénario et identifier la nature de chaque composant : appris, adapté ou à règles.
2. Réaliser l’exercice 2 sur une copie, sans modifier le corpus distribué.
3. Expliquer le dénominateur du résultat et une limite.

Les données sont sous `examples/week10/data`; le code commun sous `examples/week10/nlp.py`; les tests sous `tests/week10`. Lire [QUICKSTART](../../../../docs/week10/QUICKSTART.md) pour les dépendances, artefacts et commandes.

Les modèles entraînés sont conservés en mémoire dans chaque processus pour accélérer les notebooks. Une modification des données demande un nouveau processus ou de vider explicitement les caches trained/specialized avant de réentraîner. Les correctifs du notebook enseignant s’exécutent après le lab élève.


# Exercices

Conserver une première réponse avant les corrigés.

## Exercice 1 — Lire le budget

Pourquoi un résumé peut-il contenir moins de deux phrases alors que max_sentences=2 ?

Livrable : réponse argumentée avec calcul, sortie ou test.

## Exercice 2 — Interpréter le proxy

Un F1 lexical élevé avec la référence prouve-t-il la couverture des faits critiques ?

Livrable : réponse argumentée avec calcul, sortie ou test.

## Exercice 3 — Préserver une négation

Pourquoi éviter de couper la fin d’une phrase pour respecter le budget ?

Livrable : réponse argumentée avec calcul, sortie ou test.


# Entretien

Répondre en deux minutes par question avec un exemple et une limite.

## Question 1

Pourquoi entraîner TF-IDF sur les phrases du document est-il différent d’une fuite de labels ?

## Question 2

Que mesurer pour un résumé métier ?


# Challenge

Construire un texte où une phrase très répétitive domine le vocabulaire tandis qu’une phrase courte porte une condition essentielle. Inspecter le choix et proposer une grille d’évaluation de l’omission.

## Critères

Protocole reproductible, invariant explicite et interprétation honnête du résultat, y compris sans amélioration.


# Corrigé des exercices

## Exercice 1

Le budget en mots peut exclure les autres phrases ou le texte peut ne contenir qu’un segment. Le maximum est une borne, pas une obligation de remplissage.

## Exercice 2

Non. Les mots communs peuvent masquer une négation ou une relation incorrecte. Le proxy est un diagnostic parmi d’autres; annoter les unités d’information et les erreurs.

## Exercice 3

La coupure peut retirer une condition ou la portée d’une négation et rendre les offsets associés incohérents. Le lab garde des phrases entières.


# Réponses d’entretien

## Question 1

C’est un calcul local non supervisé sur le texte disponible à l’inférence. Il ne regarde ni la référence ni les étiquettes de test pour apprendre une sélection supervisée.

## Question 2

Les informations attendues, omissions critiques, contradictions, support dans la source, concision et utilité pour l’utilisateur. Les métriques lexicales seules sont insuffisantes.


# Solution du challenge

Comparer le résumé à une liste d’unités attendues plutôt qu’aux seuls mots. Marquer la condition comme critique et faire relire son absence. Ne pas ajuster la référence pour rendre le sélecteur meilleur. Les cellules ci-dessous vérifient d’abord les garanties effectivement implémentées : budget et slices.

## Vérification exécutable

Exécuter après les cellules du lab élève.

In [ ]:
from examples.week10.nlp import summary_overlap
small = summarize("Une phrase trop longue pour ce budget.", word_budget=1)
assert small["status"] == "budget_exhausted" and small["summary"] == ""
assert summary_overlap("a b", "a c")["f1"] == .5
assert summary_overlap("", "a c")["f1"] == 0
print("Budget et proxy lexical distingués de la fidélité métier")

# Revue pédagogique — J3

## Acquis

- Sélectionner des phrases originales sous un budget de mots.
- Conserver leurs positions et leur ordre source.
- Distinguer similarité lexicale, couverture d’information et fidélité.

## Barème indicatif sur 10

- 3 : concepts reliés au contrat métier.
- 3 : reproduction et interprétation du lab.
- 2 : exercices justifiés.
- 2 : challenge et limite de mesure.

Le parcours essentiel peut être terminé sans le challenge. Une confusion entre score logiciel et qualité générale, tête linéaire et LoRA, mention et assertion, ou masquage et anonymisation doit être corrigée avant la revue finale. Vérifier en particulier : La coupure peut retirer une condition ou la portée d’une négation et rendre les offsets associés incohérents. Le lab garde des phrases entières.


# Références — Jour 3

Sources primaires consultées le 20 octobre 2026. Le corpus et les annotations du lab sont fictifs; les seuils sont pédagogiques.

- [scikit-learn 1.8 : pièges et fuite de données](https://scikit-learn.org/1.8/common_pitfalls.html).
- [SGDClassifier 1.8](https://scikit-learn.org/1.8/modules/generated/sklearn.linear_model.SGDClassifier.html) — log_loss et partial_fit.
- [TfidfVectorizer 1.8](https://scikit-learn.org/1.8/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html).
- [Neighborhood Components Analysis 1.8](https://scikit-learn.org/1.8/modules/generated/sklearn.neighbors.NeighborhoodComponentsAnalysis.html) — apprentissage métrique supervisé.
- [spaCy : entités nommées](https://spacy.io/usage/linguistic-features#named-entities) — vocabulaire et conventions; spaCy n’est pas utilisé dans le lab obligatoire.
- [PEFT : LoRA](https://huggingface.co/docs/peft/package_reference/lora) — extension Transformer optionnelle, non exécutée.

Lire en priorité les références du thème étudié. Les concepts communs à plusieurs bibliothèques ne rendent pas leurs objets ou artefacts interchangeables. Les dépendances obligatoires correspondent à l’environnement testé, les dépendances LoRA restent à verrouiller pour l’expérience optionnelle.
